# CNN-Transformer
_An i-Python jupyter notebook to test my pipeline_

In [1]:
import numpy as np
import matplotlib as plt
import math
import mne 
from mne.io import concatenate_raws, read_raw_edf
from mne.datasets import eegbci
from mne.preprocessing import ICA
from sklearn.model_selection import train_test_split
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings

/opt/miniconda3/envs/dl26/lib/python3.12/site-packages/mne/externals/tempita/__init__.py:35: DeprecationWarning: 'cgi' is deprecated and slated for removal in Python 3.13
  import cgi


### Dataset.py
_preprocessing, loading the data of each subject_

In [ ]:
"""A plotted analysis of the dataset can be found in a 
separate Jupyter notebook in the folder 'notebooks'."""

class PreprocessedDataset(Dataset):
    def __init__(self, dataset_path, subject_ids=None, preload=True, filter_freqs=(1.0, 50.0), baseline=None):
        self.dataset_path = dataset_path
        self.subject_ids = subject_ids if subject_ids else range(1, 109) # all subjects
        self.preload = preload
        self.filter_freqs = filter_freqs
        self.baseline = baseline
        self.epochs_list = []

    def load_and_preprocess(self):
        # Empty lists to store the path for each subject
        X_subjects = []
        y_subjects = []

        for subject_id in self.subject_ids:
            # Loading the raw data file for each subject's directory
            raw, events, _ = eegbci.load_data(subject_id, runs=self.runs)
            # Overview of 10-20 montage: https://soft-dynamics.de/pdf/Int1020Syst.pdf 
            raw.set_montage(mne.channels.make_standard_montage('standard_1020'))

            # Apply bandpass filter with values defined in init fct
            raw.filter(l_freq=self.filter_freqs[0], h_freq=self.filter_freqs[1])

            # Apply ICA to remove artifacts, if necessary
            ica = ICA(num_components=20, random_state=97)
            ica.fit(raw)
            ica.apply(raw)

            # Epoching to get specific time windows of continuous temporal data
            event_id = {"left": 2, "right": 3}
            epochs = mne.Epochs(raw, events=events, event_id=event_id, 
                                tmin=0, tmax=4, baseline=self.baseline, preload=self.preload)
            self.epochs_list.append(epochs)

            # Raw EEG data for all epochs as NumPy array
            X_subjects.append(epochs.get_data) # (n_epochs, n_channels, n_samples)
            y_subjects.append(epochs.events[:, 2]) # 2nd column = event labels/IDs (our class we want to predict)
            # Third column in MNE is event_id since second column is filled with 0 and first is the sample

        # Concatenate all data from subjects for training
        X = np.concatenate(X_subjects, axis=0)
        y = np.concatenate(y_subjects, axis=0)

        # Raw event IDs do not start at 0 which PyTorch classification losses expect 
        # -> mapping to (0, 1) binary scale
        label_map = {
            2: 0,
            3: 1
        }

        # Define our target vector y
        # T1 = 0 and T2 = 1
        y = np.array([label_map[label] for label in epochs.events[:, -1]])

        return X, y

    # Knowing the number of elements in dataset
    def __len__(self): # useful for splitting dataset into training and validation set or calculating metrics
        return sum(len(epochs) for epochs in self.epochs_list)

    # Allows to access elements of dataset by index like in standard array/list
    def __getitem__(self, idx):
        for epochs in self.epochs_list:
            if idx < len(epochs):
                data = epochs.get_data()[idx]
                label = epochs.events[idx][2] - 1
                return data, label
            idx -= len(epochs)
        raise IndexError(f'Index {idx} is out of bounds')

    # Function for plotting PSDs quickly etc. (may be delayed later)
    def get_epoch(self, idx):
        """Retrieve specific epoch and its label"""
        total_len = 0
        for i, epochs in enumerate(self.epochs_list):
            if idx >= total_len and idx < total_len + len(epochs):
                epoch_idx = idx - total_len
                return self.epochs_list[i][epoch_idx]
            total_len += len(epochs)
        raise IndexError(f'Index {idx} is out of bounds')

### Datamodule.py
_creating data loaders to organize the data into small batches_

In [ ]:
def create_dataloaders(X, y):
    subjects = list(range(1, 109)) # 108 participants

    # 80 % of subjects used for training, 20 % for testing
    train_subjects, test_subjects = train_test_split(subjects, test_size=0.2, random_state=42)

    X_train = []
    y_train = []

    for subject in train_subjects:
        X_subj, y_subj = PreprocessedDataset.load_and_preprocess(subject)
        X_train.append(X_subj)
        y_train.append(y_subj)

    X_test = []
    y_test = []

    for subject in test_subjects:
        X_subj, y_subj = PreprocessedDataset.load_and_preprocess(subject)
        X_test.append(X_subj)
        y_test.append(y_subj)
    
    train_dataset = PreprocessedDataset(X_train, y_train)
    test_dataset = PreprocessedDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) 
    # Some other projects use batch size of 16 with the PhysioNet dataset
    test_loader = DataLoader(test_dataset, batch_size=32)

    return train_loader, test_loader

### Model.py
_creating the different classes for the baseline model and main model components_

In [ ]:
"""A small and simple baseline model to compare to"""

class BaselineMLP(nn.Module):

    def __init__(self, input_dim, num_classes, hidden_dim=64, dropout_rate=0.5) -> None:
        # EEG datasets small & noisy -> strong dropout suggested
        # 64 channels/electrodes -> 1:1 mapping
        # Call constructor of parent class (nn.Module)
        super(BaselineMLP, self).__init__()
        
        self.layer_1 = nn.Linear(input_dim, hidden_dim)
        self.layer_2 = nn.Linear(hidden_dim, hidden_dim // 2)
        # enforce dense representations (hidden_dim//2)
        self.layer_out = nn.Linear(hidden_dim // 2, num_classes)
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()
        
        def forward(self, x):
            x = self.relu(self.layer_1(x))
            x = self.dropout(x)
            x = self.relu(self.layer_2(x))
            x = self.dropout(x)
            x = self.layer_out(x)
            return x

        """ ALTERNATIVE CODE:
                self.network = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU()
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim // 2, num_classes)
                )
        
                def forward(self, x):
                    return self.network(x)"""

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, emb_dim, max_patches):
        super(PositionalEncoding, self).__init__()
        self.emb_dim = emb_dim

        # Positional encoding matrix
        pe = torch.zeros(max_patches, emb_dim)

        # Sinusoidal positional encoding
        # Position indices
        pos = torch.arange(0, max_patches, dtype=torch.float).unsqueeze(1) # (max_patches, 1)

        # Division term (tensor of even indices since pose alternate between sin/cos)
        div_term = torch.exp(torch.arange(0, emb_dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / emb_dim))

        pe[:, 0::2] = torch.sin(pos * div_term) # Every second column starting from index 0
        pe[:, 1::2] = torch.cos(pos * div_term) # For all odd indices cosinusoidal values

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :] # (batch_size, seq_len, d_model)
        return x

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_channels, num_classes=2, emb_dim=128):
        super(CNN, self).__init__()

        # Choice between Conv1d and Conv2d was hard -> so why 1d?
        # Look into kernel size and stride/padding choices!
        self.conv_layers == nn.Sequential(
            nn.Conv1d(num_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )

        self.flatten = nn.Flatten()
        self.embedding_layer = nn.Linear(128 * (num_classes // 4), emb_dim)

        def forward(self, x):
            x = self.conv_layers(x)
            x = self.flatten(x)
            x = self.embedding_layer(x)
            return x

In [ ]:
# Transformer components

class ResidualConnection(nn.Module):
    def __init__(self, block, norm=nn.LayerNorm, dropout=0.1):
        super(ResidualConnection, self).__init__()
        self.block = block
        self.norm = norm()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.norm(x + self.block(x))
        x = self.dropout(x)
        return x


class FeedForwardBlock(nn.Sequential):
    def __init__(self, input_dim, emb_dim, dropout_rate):
        """
        emb_dim: dim of model's in- & output
        expansion: dim of inner layer in FFN"""

        super(FeedForwardBlock, self).__init__()
        self.linear = nn.Linear(input_dim, input_dim * emb_dim)
        self.silu = nn.SiLU()
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(input_dim * emb_dim, input_dim)

    def forward(self, x):
        x = self.linear(x)
        x = self.relu(x) # Switch to SiLU to experiment
        x = self.dropout(x)
        x = self.out(x)
        return x

In [ ]:
class EncoderBlock(nn.Sequential):
    def __init__(self, input_dim, emb_dim, num_heads, dropout_rate=0.0):
        """dim_ff: dimension for feedforward pass"""
        super(EncoderBlock, self).__init__()

        self.attention = nn.MultiheadAttention(emb_dim, num_heads, dropout_rate)
        self.ffn = FeedForwardBlock(emb_dim, expansion, dropout_rate)
        self.residual1 = ResidualConnection(self.attention)
        self.residual2 = ResidualConnection(self.ffn)
        self.residual3 = ResidualConnection(self.ffn)

    def forward(self, x, mask=None):
        x = self.residual1(x)
        x = self.residual2(x)
        x = self.residual3(x)
        return x

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, eeg_channel, dropout_rate=0.1):
        super(MLPClassifier, self).__init__()

        self.linear_in = nn.Linear(eeg_channel * 2, eeg_channel // 2)
        self.linear_out = nn.Linear(eeg_channel // 2, 1)
        self.relu = nn.ReLU(True)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.linear_in(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear_out(x)
        return x

In [ ]:
# Final model

class EEGClassifier(nn.Module):
    def __init__(self, input_dim=64, emb_dim=128, max_patches=, 
                 num_channels=64, num_heads=, dropout_rate=0.5, eeg_channel=):
        """
        num_channels:   number of EEG channels/model dimension, usually 64
        """
        super(EEGClassifier, self).__init__()

            """
            Positional Encoding
            CNN
            EncodingBlock
            MLPClassifier
            """
    
        self.positional_encoding = PositionalEncoding(emb_dim, max_patches) 
        self.cnn = CNN(num_channels)
        self.transformer = EncoderBlock(input_dim, emb_dim, num_heads, dropout_rate)
        self.mlp = MLPClassifier(eeg_channel) # Assuming pooling reduces patches by half

    def forward(self, x):
        """
        Args: x: Input EEG data (batch_size, num_channels, num_time_points)
        Returns: logits for classification
        """
        x = self.positional_encoding(x)
        x = self.cnn(x)             # (batch_size, num_channels, n_samples)
        x = self.permute(2, 0, 1)   # (n_samples, batch_size, num_channels)
        x = self.transformer(x)     # (n_samples, batch_size, num_channels)
        x = self.mlp(x)
        
        return x.squeeze() # Squeeze to remove single dim for binary classification

### Train.py
_Loading and preprocessing the data, passing it through the baseline MLP and then main model, optimizing via the Adam optimizer and training both the baseline and main model._

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuddn.deterministic = True

# Preprocess data
preprocessor = PreprocessedDataset()
X, y = preprocessor.load_and_preprocess()
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

# Create train and test dataloaders
train_loader, test_loader = create_dataloaders(X, y)

# Training loop for main model
num_epochs = 8 # number of epochs

#### Hyperparameter Tuning of the baseline MLP
1. Size of hidden dimension:
* hidden_dim = input_dim - test if a linear classifier like Logistic Regression would be sufficient
* hidden_dim < input_dim (compression) - assumption of highly correlated spectral features due to simple task 
* hidden_dim > input_dim (expansion) - assume a highly complex relationship between features and output class (increases risk of overfitting)

-> compression seems the most intuitive solution to me since the underlying task was not too difficult

In [ ]:
model_baseline_linear = BaselineMLP(input_dim=64, num_classes=2, hidden_dim=64)
model_baseline_compr = BaselineMLP(input_dim=64, num_classes=2, hidden_dim=32)
model_baseline_expansion = BaselineMLP(input_dim=64, num_classes=2, hidden_dim=128)

#### Training the baseline MLP

In [ ]:
# Function to train any of the following models

def train_model(model, x, num_epochs, dataloader, optimizer):
    for epoch in range(num_epochs):
        for batch in dataloader:
            model.train()
            optimizer.zero_grad()
            pred = model(x)
            loss = nn.CrossEntropyLoss(pred, x)
            loss.backward()
            optimizer.step()
        print(loss.item)

In [ ]:
model_baseline = BaselineMLP(input_dim=64, num_classes=2) 
# hidden_dim and dropout could also be modified if desired (and then plotted)
optimizer_baseline = optim.Adam(model_baseline.parameters(), lr=0.001)
train_model(model_baseline, x, num_epochs, train_loader, optimizer_baseline)

"""
If the train_model fct does not work:

for epoch in range(num_epochs):
    for batch in dataloader:
        train(model_baseline, train_loader)
        optimizer_baseline.zero_grad()
        pred = model_baseline(x)
        loss = nn.CrossEntropyLoss(pred, x)
        loss.backward()
        optimizer_baseline.step()
    print(loss.item)"""

#### Training the main CNN-Transformer

In [ ]:
# model_main = EEGClassifier()

optimizer_main = optim.Adam(model_main.parameters(), lr=0.001)
# train_model(model_main, x, num_epochs, Prepro, train_loader, optimizer_main)

### Evaluate.py
_compare the baseline MLP vs. CNN-Transformer performance using accuracy and plotting other performance metrics_

In [ ]:
def accuracy(model, test_loader):
    model.load_state_dict(torch.load("checkpoint.pt"))
    model.eval()

    correct = 0

    with torch.no_grad():
        for x, y in test_loader:
            output = model(x)
            pred = output.argmax(dim=1)
            correct += (pred==y).sum()

    acc = correct / len(test_loader.dataset)
    return {'acc': acc} 
    # Change dictionary item output perhaps

In [ ]:
metrics = {
    'Baseline Linear': accuracy(model_baseline_linear, test_loader),
    'Baseline Compression': accuracy(model_baseline_compr, test_loader),
    'Baseline Expansion': accuracy(model_baseline_expansion, test_loader)
}

fig, ax = plt.subplots(figsize=(13, 5))
for model, metric in metrics.items():
    ax.plot([model], [metric['acc']], marker='o', label=model)

ax.set_xlabel('Model')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Model Performance Comparison')
ax.legend()
plt.tight_layout()
plt.show()

"""
If accuracy fct does not work:


model_baseline.load_state_dict(torch.load("checkpoint.pt"))
model_baseline.eval()

correct = 0

with torch.no_grad():
    for x, y in test_loader:
        output_baseline = model_baseline(x)
        pred_baseline = output_baseline.argmax(dim=1)
        correct_baseline += (pred_baseline==y).sum()

acc_baseline = correct_baseline / len(test_loader.dataset)
print(f"Accuracy of the baseline MLP: {acc_baseline}")"""

In [ ]:
"""model_main = EEGClassifier() # add params here
accuracy(model_main, test_loader)"""